# Customer Data Cleaning Project

**Goal:** take a messy 50-row customer dataset and turn it into a clean, analysis-ready file using Python and pandas.

**Dataset:** `messy_customers.csv` (5 columns: `customer_name`, `city`, `signup_date`, `age`, `monthly_spend`)

**Problems found:** duplicate rows, missing values, inconsistent text (case, spaces, spellings), dates in 4 different formats, text mixed into numeric columns, and impossible or extreme values.

**Approach:** keep the original in `df`, do all cleaning on a copy called `df_clean`, and compare before and after at the end.

In [ ]:
import pandas as pd

df = pd.read_csv("messy_customers.csv")
df_clean = df.copy()   # df = untouched original, df_clean = the copy we clean

df.head(10)

## 1. Inspect the data
First look at the size, data types, missing values, duplicates and the number of unique cities.

In [ ]:
print("Shape:", df.shape)
print()
df.info()
print("\nMissing values per column:")
print(df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("Unique cities:", df["city"].nunique())

**What this shows:** `age` and `monthly_spend` are stored as text (object) because of values like "27 years" and "€57.22". There are 2 duplicate rows, a few missing values, and 11 different city spellings when there should be 6.

## 2. Remove duplicate rows

In [ ]:
before = len(df_clean)
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
print(f"Rows: {before} -> {len(df_clean)}  ({before - len(df_clean)} duplicates removed)")

## 3. Clean customer names
Remove extra spaces, use consistent capitalisation, and label blank names as "Unknown".

In [ ]:
df_clean["customer_name"] = df_clean["customer_name"].str.strip().str.title()
df_clean["customer_name"] = df_clean["customer_name"].fillna("Unknown")

df_clean["customer_name"].head(10)

## 4. Clean city names
Fix spaces and capitalisation, then merge the spelling variants (Tirane -> Tirana, Durrës -> Durres).

In [ ]:
print("Before:", df_clean["city"].nunique(), "unique values")

df_clean["city"] = df_clean["city"].str.strip().str.title()
df_clean["city"] = df_clean["city"].replace({"Tirane": "Tirana", "Durrës": "Durres"})
df_clean["city"] = df_clean["city"].fillna("Unknown")

print("After:", df_clean["city"].nunique(), "unique values (including 'Unknown')")
print()
print(df_clean["city"].value_counts())

## 5. Fix the dates
The dates come in 4 formats (`2024-03-05`, `2025/01/27`, `14-08-2024`, `March 5, 2025`). I try each format explicitly instead of using `format="mixed"`, because a date like `05-03-2025` is ambiguous (5 March or 3 May?).

In [ ]:
s = df_clean["signup_date"].str.strip()

parsed = pd.to_datetime(s, format="%Y-%m-%d", errors="coerce")
for fmt in ["%Y/%m/%d", "%d-%m-%Y", "%B %d, %Y"]:
    parsed = parsed.fillna(pd.to_datetime(s, format=fmt, errors="coerce"))

failed = s.notna() & parsed.isna()
print("Dates that could not be parsed:", failed.sum())

df_clean["signup_date"] = parsed
print("Missing dates (blank in the original):", df_clean["signup_date"].isna().sum())
print("Data type:", df_clean["signup_date"].dtype)

## 6. Clean the age column
Convert text to numbers, treat impossible ages (outside 18-100) as missing, then fill the missing values with the median.

In [ ]:
# keep only digits, minus sign and dot ("27 years" -> "27", "N/A" -> "")
age = df_clean["age"].astype(str).str.replace(r"[^\d.\-]", "", regex=True)
age = pd.to_numeric(age, errors="coerce")

# realistic range for this dataset: 18-100
invalid = (age < 18) | (age > 100)
print("Impossible ages found:", invalid.sum(), "->", age[invalid].tolist())
age = age.mask(invalid)

median_age = age.median()
print("Missing ages to fill:", age.isna().sum(), "-> median =", median_age)
df_clean["age"] = age.fillna(median_age).round().astype(int)

## 7. Clean monthly spend
Remove currency symbols and units, turn comma decimals into dots, convert to numbers, detect outliers with the IQR rule, then fill the missing values with the median.

In [ ]:
spend = (
    df_clean["monthly_spend"].astype(str)
    .str.replace("€", "", regex=False)
    .str.replace("EUR", "", regex=False)
    .str.replace(",", ".", regex=False)
    .str.strip()
)
spend = pd.to_numeric(spend, errors="coerce")

# outliers with the IQR rule
q1, q3 = spend.quantile(0.25), spend.quantile(0.75)
upper = q3 + 1.5 * (q3 - q1)
outliers = spend > upper
print("Outliers found:", outliers.sum(), "(upper limit =", round(upper, 2), ")", spend[outliers].tolist())
spend = spend.mask(outliers)

median_spend = spend.median()
print("Missing spend to fill:", spend.isna().sum(), "-> median =", round(median_spend, 2))
df_clean["monthly_spend"] = spend.fillna(median_spend).round(2)

## 8. Final checks

In [ ]:
print("Missing values per column:")
print(df_clean.isna().sum())
print("\nDuplicate rows:", df_clean.duplicated().sum())
print()
df_clean.info()

## 9. Before vs after

In [ ]:
summary = pd.DataFrame({
    "Before": [
        len(df),
        df.duplicated().sum(),
        df.isna().sum().sum(),
        df["city"].nunique(),
    ],
    "After": [
        len(df_clean),
        df_clean.duplicated().sum(),
        df_clean.isna().sum().sum(),
        df_clean.loc[df_clean["city"] != "Unknown", "city"].nunique(),
    ],
}, index=["Rows", "Duplicate rows", "Missing values", "Unique cities (excl. Unknown)"])

summary

## 10. Save the clean dataset

In [ ]:
df_clean.to_csv("clean_customers.csv", index=False)
print("Saved clean_customers.csv with", len(df_clean), "rows")

df_clean.head(10)

## Summary of decisions
- **Duplicates:** removed exact duplicate rows.
- **Text:** trimmed spaces, applied title case, merged spelling variants of the same city.
- **Missing names and cities:** labelled "Unknown" so the rows are kept.
- **Dates:** converted all 4 formats to a single datetime type. The one blank date stays missing because a signup date can't be guessed.
- **Age:** converted to numbers, impossible values (below 18 or above 100) treated as missing, then filled with the median.
- **Monthly spend:** removed symbols and units, fixed comma decimals, treated the extreme value as an outlier (IQR rule), then filled with the median.

**Tools:** Python, pandas, Jupyter Notebook.